<a href="https://colab.research.google.com/github/mkvkanpur/hpc_book/blob/main/Jax.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# sum(a*b)

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# 1. Initialize data (JAX uses its own device-backed arrays)
n = 1_000_000
key = jax.random.PRNGKey(0) # JAX handles randomness explicitly
a = jax.random.normal(key, (n,))
b = jax.random.normal(key, (n,))

# 2. Define the mathematical operation
def dot_product(x, y):
    return jnp.sum(x * y)

# 3. Apply the JIT transformation
# This compiles the Python code into an optimized GPU/TPU kernel
fast_dot = jax.jit(dot_product)

# Run it! (The first call includes compilation time)
result = fast_dot(a, b)

# Block until finished to get accurate timing (JAX is asynchronous)
result.block_until_ready()

print(f"JAX Dot Product Result: {result}")


JAX Dot Product Result: 997774.375
JAX Dot Product Result: [2.6329675  4.1016974  0.18800414 ... 0.2115733  0.02358132 0.2678787 ]


In [ ]:
import jax
import jax.numpy as jnp

# Define the core 'Sovereign Row' logic
def row_dot(a_row, b_row):
    return jnp.sum(a_row * b_row)

# Lift the function to handle a Batch (Vertical Parallelism)
# This replaces manual 'tile_id' and 'local_id' loops
batch_dot = jax.vmap(row_dot)

# The Full Reduction: Map across rows, then sum the results
@jax.jit
def sovereign_dot_product(A, B):
    # Phase 1: vmap executes 1024 dot products in parallel
    row_results = batch_dot(A, B)
    # Phase 2: Final collapse into a single scalar
    return jnp.sum(row_results)

# Execution on a 1024x1024 Manifold (12 GB Total Data)
N = 1024
key = jax.random.PRNGKey(0)
A = jax.random.normal(key, (N, N))
B = jax.random.normal(key, (N, N))

# Achieving the 3.5 ms Bandwidth Limit
result = sovereign_dot_product(A, B)
print(result)

1046626.6


# y=Ax

In [ ]:
import jax
import jax.numpy as jnp

# Define the core 'Sovereign Row' logic
# a_row: a single row of the matrix (1D array)
# x: the full vector to be dotted against (1D array)
def row_dot(a_row, x):
    return jnp.sum(a_row * x)

# Lift the function to handle Matrix-Vector multiplication
# in_axes=(0, None) means:
# - Map over the 0-th axis (rows) of the first argument (Matrix A)
# - Keep the second argument (Vector x) whole for every call
batch_dot = jax.vmap(row_dot, in_axes=(0, None))

# JIT compile to fuse the mapping and summation into one GPU kernel
@jax.jit
def mat_vec_product(A, x):
    return batch_dot(A, x)

# Execution on a 1024x1024 Manifold
N = 32
key = jax.random.PRNGKey(42)

Matrix_A = jax.random.normal(key, (N, N))
vector_x = jax.random.normal(key, (N,))

# This executes at the 3.5 ms hardware bandwidth limit
result = mat_vec_product(Matrix_A, vector_x)
print(result)

[24.757122   -3.6344657   4.8118625  -8.36869     1.9498956  -5.8683844
 -6.534626    0.63047165  2.712511    3.3620389  -1.9465117   6.2772408
  5.1532235  -4.998759   -3.0583997  -6.1924777   0.366652   -0.34393376
 -4.492004   -6.04481    -0.5699376   3.0316677  -7.5910306   3.1692276
  0.9132905  -1.0058215   1.8434484  -1.2929274   2.588693   -0.79371935
  0.8671037   0.6863174 ]


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# 1. Setup dimensions
m, n = 5000, 5000
key = jax.random.PRNGKey(42)
k1, k2 = jax.random.split(key)

# 2. Initialize Matrix A and Vector x
# JAX arrays are stored on the GPU/TPU by default if available
A = jax.random.normal(k1, (m, n))
x = jax.random.normal(k2, (n,))

# 3. Define the y = Ax function
def matvec(A, x):
    # You can use jnp.dot(A, x) or the @ operator
    return A @ x

# 4. JIT Compile for high performance
# This triggers XLA to optimize the linear algebra
fast_matvec = jax.jit(matvec)

# Warm-up (Compiles the function)
y = fast_matvec(A, x)

# Timing the execution (Blocking ensures we measure actual GPU time)
y.block_until_ready()
print(f"Shape of y: {y.shape}")
print("Matrix-vector multiplication successful and optimized!")

Shape of y: (5000,)
Matrix-vector multiplication successful and optimized!


# N-body problem

In [ ]:
import jax
import jax.numpy as jnp

# 1. Physics Constants (G = 1 for normalized units)
G = 1.0
SOFTENING = 0.1  # Prevents infinite force at zero distance

@jax.jit
def compute_forces(positions, masses):
    """
    Computes the gravitational force on each particle.
    positions: (N, 3) array
    masses: (N,) array
    """

    def body_to_body_force(pos_i, pos_j, m_j):
        """Force exerted on particle i by particle j"""
        dr = pos_j - pos_i
        dist_sq = jnp.sum(dr**2) + SOFTENING**2
        force_mag = G * m_j / (dist_sq * jnp.sqrt(dist_sq))
        return force_mag * dr

    # Vectorize over j (the sources of gravity)
    # This computes the force on one particle 'i' from ALL particles 'j'
    def total_force_on_i(pos_i):
        forces = jax.vmap(body_to_body_force, in_axes=(None, 0, 0))(pos_i, positions, masses)
        return jnp.sum(forces, axis=0)

    # Vectorize over i (the targets of gravity)
    # This computes the force for EVERY particle in the system
    return jax.vmap(total_force_on_i)(positions)

# --- Simulation Setup ---
N = 1000
key = jax.random.PRNGKey(0)
pos_key, mass_key = jax.random.split(key)

# Random positions and masses
positions = jax.random.normal(pos_key, (N, 3))
masses = jax.random.uniform(mass_key, (N,))

# Compute forces (First call triggers JIT compilation)
forces = compute_forces(positions, masses).block_until_ready()
print(f"Computed forces for {N} particles. Shape: {forces.shape}")

Computed forces for 1000 particles. Shape: (1000, 3)


# 1D derivative

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit

# 1. The Pure Math Function
def central_diff(f, dx):
    # jnp.roll shifts the array to get neighbors
    # f_right = f[i+1], f_left = f[i-1]
    f_right = jnp.roll(f, -1)
    f_left  = jnp.roll(f, 1)

    # Interior: Central Difference
    df = (f_right - f_left) / (2.0 * dx)

    # 2. Manual Boundary Correction
    # Forward at start, Backward at end
    df = df.at[0].set((f[1] - f[0]) / dx)
    df = df.at[-1].set((f[-1] - f[-2]) / dx)

    return df

# 3. The HPC Transformation (The JIT)
# This compiles the function into a fused XLA kernel
fast_diff = jit(central_diff)

# --- Execution ---
N = 2**16
dx = 1.0 / N
x = jnp.linspace(0, 1, N)
f = jnp.sin(2 * jnp.pi * x)

# First call triggers compilation; second call is near-instant
df = fast_diff(f, dx).block_until_ready()